[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/24.%20The%20Static%20Truck%20Appointment%20System%20Problem/P24-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 24. The Static Truck Appointment System Problem

## Tier 2: The Classic Heuristic (Python Implementation)

### Goal
Implement the Earliest Deadline First (EDF) constructive heuristic to efficiently solve the Static Truck Appointment System Problem. This approach provides near-optimal solutions with computational efficiency suitable for real-time terminal operations.

### Key assumptions
- Requests are processed in order of preferred time (earliest deadline first)
- Greedy assignment strategy with capacity checking
- No backtracking or reassignment of previously placed requests
- Simple and fast algorithm suitable for real-time operations
- Limited optimization focus on immediate feasibility

### Approach (step-by-step)
1. **Sort requests by preferred time** (earliest deadline first)
2. **Iterate through sorted requests** in chronological order
3. **Check capacity constraints** for each time slot
4. **Assign to preferred slot** if capacity available
5. **Find nearest available slot** if preferred slot is full
6. **Calculate total deviation penalty** for the solution
7. **Analyze performance** and compare with optimal solutions

### What to look for in the results
- Assignment speed and computational efficiency
- Solution quality compared to optimal MIP solution
- Deviation patterns and their causes
- Algorithm complexity analysis and scalability
- Trade-offs between speed and optimality

### Concrete example (from the source)
The EDF heuristic operates on the principle that accommodating time-sensitive requests first leads to better overall system performance. By sorting requests according to their preferred arrival times and processing them sequentially, the algorithm minimizes the total deviation penalty while maintaining computational efficiency.

**Expected Output:**
```
EDF Assignment Results:
Request 1: Assigned to slot 0 (preferred: 0)
Request 2: Assigned to slot 0 (preferred: 0)
Request 3: Assigned to slot 1 (preferred: 1)
Request 4: Assigned to slot 1 (preferred: 1)
Request 5: Assigned to slot 2 (preferred: 2)
Request 6: Assigned to slot 3 (preferred: 3)
Total Deviation Penalty: 0
```

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for heuristic implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully for EDF heuristic implementation")

Libraries imported successfully for EDF heuristic implementation


In [ ]:
# Define data structures (reuse from Tier 1)
@dataclass
class TruckRequest:
    """Represents a truck appointment request"""
    id: int
    preferred_time: int  # Preferred time slot (0-indexed)
    weight: float  # Priority weight
    service_type: str = 'standard'

@dataclass
class TimeSlot:
    """Represents a time slot with capacity constraints"""
    id: int
    capacity: int  # Maximum trucks that can be processed
    current_load: int = 0  # Current assigned trucks

@dataclass
class Gate:
    """Represents a processing gate"""
    id: int
    capacity_per_slot: int = 4
    available: bool = True

print("Data structures defined for EDF heuristic")

Data structures defined for EDF heuristic


In [ ]:
try:
    # Create test instances including the concrete example
    def create_test_instances():
        """Create multiple test instances for evaluation"""
        
        instances = {}
        
        # Instance 1: Concrete example from source
        requests1 = [
            TruckRequest(1, 0, 1.0),
            TruckRequest(2, 0, 1.0),
            TruckRequest(3, 1, 2.0),
            TruckRequest(4, 1, 1.0),
            TruckRequest(5, 2, 1.0),
            TruckRequest(6, 3, 2.0)
        ]
        time_slots1 = [
            TimeSlot(0, 2),
            TimeSlot(1, 2),
            TimeSlot(2, 1),
            TimeSlot(3, 1)
        ]
        instances['concrete'] = (requests1, time_slots1)
        
        # Instance 2: Larger test case
        requests2 = [
            TruckRequest(i, np.random.randint(0, 6), np.random.uniform(0.5, 2.0))
            for i in range(1, 13)
        ]
        time_slots2 = [TimeSlot(i, 3) for i in range(6)]
        instances['large'] = (requests2, time_slots2)
        
        # Instance 3: Stress test with capacity constraints
        requests3 = [
            TruckRequest(1, 0, 1.0), TruckRequest(2, 0, 1.0), TruckRequest(3, 0, 1.0),
            TruckRequest(4, 1, 1.0), TruckRequest(5, 1, 1.0), TruckRequest(6, 1, 1.0),
            TruckRequest(7, 2, 1.0), TruckRequest(8, 2, 1.0), TruckRequest(9, 2, 1.0)
        ]
        time_slots3 = [TimeSlot(i, 2) for i in range(3)]
        instances['stress'] = (requests3, time_slots3)
        
        return instances
    
    test_instances = create_test_instances()
    print(f"Created {len(test_instances)} test instances:")
    for name, (reqs, slots) in test_instances.items():
        print(f"  {name}: {len(reqs)} requests, {len(slots)} time slots")
except Exception as e:
    print(f'Error in cell: {e}')

Created 3 test instances:
  concrete: 6 requests, 4 time slots
  large: 12 requests, 6 time slots
  stress: 9 requests, 3 time slots


In [ ]:
# Earliest Deadline First (EDF) Heuristic Implementation
class EarliestDeadlineFirst:
    """EDF Heuristic for Static Truck Appointment System"""
    
    def __init__(self, requests: List[TruckRequest], time_slots: List[TimeSlot]):
        self.requests = requests
        self.time_slots = time_slots
        self.assignments = {}
        self.execution_time = 0
        self.total_deviation = 0
        self.weighted_deviation = 0
    
    def solve(self) -> Dict:
        """Solve using EDF heuristic"""
        start_time = time.time()
        
        # Reset time slots
        for slot in self.time_slots:
            slot.current_load = 0
        
        # Sort requests by preferred time (earliest deadline first)
        sorted_requests = sorted(self.requests, key=lambda r: r.preferred_time)
        
        # Assign each request
        for request in sorted_requests:
            assigned_slot = self._assign_request(request)
            self.assignments[request.id] = assigned_slot
        
        # Calculate performance metrics
        self._calculate_metrics()
        
        self.execution_time = time.time() - start_time
        
        return self._get_solution()
    
    def _assign_request(self, request: TruckRequest) -> int:
        """Assign a single request to the best available time slot"""
        
        # Try preferred time slot first
        preferred_slot = self.time_slots[request.preferred_time]
        if preferred_slot.current_load < preferred_slot.capacity:
            preferred_slot.current_load += 1
            return preferred_slot.id
        
        # Find nearest available slot (search forward then backward)
        # Search forward (later slots)
        for i in range(request.preferred_time + 1, len(self.time_slots)):
            slot = self.time_slots[i]
            if slot.current_load < slot.capacity:
                slot.current_load += 1
                return slot.id
        
        # Search backward (earlier slots)
        for i in range(request.preferred_time - 1, -1, -1):
            slot = self.time_slots[i]
            if slot.current_load < slot.capacity:
                slot.current_load += 1
                return slot.id
        
        # No available slot (should not happen with proper capacity planning)
        raise ValueError(f"No available slot for request {request.id}")
    
    def _calculate_metrics(self):
        """Calculate performance metrics"""
        self.total_deviation = 0
        self.weighted_deviation = 0
        
        for request in self.requests:
            assigned_slot = self.assignments[request.id]
            deviation = abs(assigned_slot - request.preferred_time)
            self.total_deviation += deviation
            self.weighted_deviation += request.weight * deviation
    
    def _get_solution(self) -> Dict:
        """Return solution dictionary"""
        return {
            'assignments': self.assignments,
            'total_deviation': self.total_deviation,
            'weighted_deviation': self.weighted_deviation,
            'execution_time': self.execution_time,
            'algorithm': 'EDF'
        }

print("EDF heuristic class defined successfully")

EDF heuristic class defined successfully


In [ ]:
try:
    # Run EDF on concrete example
    def run_concrete_example():
        """Run EDF on the concrete example from source"""
        
        print("=" * 70)
        print("EARLIEST DEADLINE FIRST HEURISTIC - CONCRETE EXAMPLE")
        print("=" * 70)
        
        requests, time_slots = test_instances['concrete']
        
        print("\nProblem Instance:")
        print(f"  Requests: {len(requests)}")
        print(f"  Time Slots: {len(time_slots)}")
        print(f"  Total Capacity: {sum(slot.capacity for slot in time_slots)}")
        
        print("\nRequest Details (sorted by preferred time):")
        sorted_requests = sorted(requests, key=lambda r: r.preferred_time)
        for req in sorted_requests:
            print(f"  Request {req.id}: Preferred slot {req.preferred_time}, Weight {req.weight}")
        
        print("\nTime Slot Capacities:")
        for slot in time_slots:
            print(f"  Slot {slot.id}: Capacity {slot.capacity}")
        
        # Run EDF heuristic
        edf = EarliestDeadlineFirst(requests, time_slots)
        solution = edf.solve()
        
        print("\n" + "="*50)
        print("EDF ASSIGNMENT RESULTS")
        print("="*50)
        
        print("\nAssignment Details:")
        for req in sorted_requests:
            assigned_slot = solution['assignments'][req.id]
            deviation = abs(assigned_slot - req.preferred_time)
            status = "✓" if deviation == 0 else f"(+{deviation})"
            print(f"  Request {req.id}: Slot {assigned_slot} (preferred: {req.preferred_time}) {status}")
        
        print(f"\nPerformance Metrics:")
        print(f"  Total Deviation: {solution['total_deviation']}")
        print(f"  Weighted Deviation: {solution['weighted_deviation']:.2f}")
        print(f"  Execution Time: {solution['execution_time']:.6f} seconds")
        
        print("\nTime Slot Utilization:")
        for slot in time_slots:
            assigned_count = sum(1 for req in requests if solution['assignments'][req.id] == slot.id)
            utilization = assigned_count / slot.capacity * 100
            print(f"  Slot {slot.id}: {assigned_count}/{slot.capacity} ({utilization:.1f}% utilized)")
        
        return solution, requests, time_slots
    
    concrete_solution, concrete_requests, concrete_slots = run_concrete_example()
except Exception as e:
    print(f'Error in cell: {e}')

EARLIEST DEADLINE FIRST HEURISTIC - CONCRETE EXAMPLE

Problem Instance:
  Requests: 6
  Time Slots: 4
  Total Capacity: 6

Request Details (sorted by preferred time):
  Request 1: Preferred slot 0, Weight 1.0
  Request 2: Preferred slot 0, Weight 1.0
  Request 3: Preferred slot 1, Weight 2.0
  Request 4: Preferred slot 1, Weight 1.0
  Request 5: Preferred slot 2, Weight 1.0
  Request 6: Preferred slot 3, Weight 2.0

Time Slot Capacities:
  Slot 0: Capacity 2
  Slot 1: Capacity 2
  Slot 2: Capacity 1
  Slot 3: Capacity 1

EDF ASSIGNMENT RESULTS

Assignment Details:
  Request 1: Slot 0 (preferred: 0) ✓
  Request 2: Slot 0 (preferred: 0) ✓
  Request 3: Slot 1 (preferred: 1) ✓
  Request 4: Slot 1 (preferred: 1) ✓
  Request 5: Slot 2 (preferred: 2) ✓
  Request 6: Slot 3 (preferred: 3) ✓

Performance Metrics:
  Total Deviation: 0
  Weighted Deviation: 0.00
  Execution Time: 0.000013 seconds

Time Slot Utilization:
  Slot 0: 2/2 (100.0% utilized)
  Slot 1: 2/2 (100.0% utilized)
  Slot 2: 1/1 

In [ ]:
# Enhanced EDF with alternative strategies
class EnhancedEDF:
    """Enhanced EDF with multiple assignment strategies"""
    
    def __init__(self, requests: List[TruckRequest], time_slots: List[TimeSlot]):
        self.requests = requests
        self.time_slots = time_slots
        self.strategies = {
            'basic': self._basic_edf,
            'weighted': self._weighted_edf,
            'capacity_balanced': self._capacity_balanced_edf
        }
    
    def solve(self, strategy: str = 'basic') -> Dict:
        """Solve using specified strategy"""
        if strategy not in self.strategies:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        start_time = time.time()
        
        # Reset time slots
        for slot in self.time_slots:
            slot.current_load = 0
        
        # Apply strategy
        assignments = self.strategies[strategy]()
        
        # Calculate metrics
        total_deviation = 0
        weighted_deviation = 0
        
        for request in self.requests:
            assigned_slot = assignments[request.id]
            deviation = abs(assigned_slot - request.preferred_time)
            total_deviation += deviation
            weighted_deviation += request.weight * deviation
        
        execution_time = time.time() - start_time
        
        return {
            'assignments': assignments,
            'total_deviation': total_deviation,
            'weighted_deviation': weighted_deviation,
            'execution_time': execution_time,
            'strategy': strategy
        }
    
    def _basic_edf(self) -> Dict:
        """Basic EDF strategy"""
        assignments = {}
        sorted_requests = sorted(self.requests, key=lambda r: r.preferred_time)
        
        for request in sorted_requests:
            assignments[request.id] = self._find_best_slot_basic(request)
        
        return assignments
    
    def _weighted_edf(self) -> Dict:
        """Weighted EDF strategy (consider weights in tie-breaking)"""
        assignments = {}
        # Sort by preferred time, then by weight (higher weight first)
        sorted_requests = sorted(self.requests, key=lambda r: (r.preferred_time, -r.weight))
        
        for request in sorted_requests:
            assignments[request.id] = self._find_best_slot_basic(request)
        
        return assignments
    
    def _capacity_balanced_edf(self) -> Dict:
        """Capacity-balanced EDF strategy"""
        assignments = {}
        sorted_requests = sorted(self.requests, key=lambda r: r.preferred_time)
        
        for request in sorted_requests:
            assignments[request.id] = self._find_best_slot_balanced(request)
        
        return assignments
    
    def _find_best_slot_basic(self, request: TruckRequest) -> int:
        """Find best slot using basic strategy"""
        # Try preferred slot first
        preferred_slot = self.time_slots[request.preferred_time]
        if preferred_slot.current_load < preferred_slot.capacity:
            preferred_slot.current_load += 1
            return preferred_slot.id
        
        # Find nearest available
        for offset in range(1, max(len(self.time_slots), request.preferred_time + 1)):
            # Check forward
            forward_idx = request.preferred_time + offset
            if forward_idx < len(self.time_slots):
                slot = self.time_slots[forward_idx]
                if slot.current_load < slot.capacity:
                    slot.current_load += 1
                    return slot.id
            
            # Check backward
            backward_idx = request.preferred_time - offset
            if backward_idx >= 0:
                slot = self.time_slots[backward_idx]
                if slot.current_load < slot.capacity:
                    slot.current_load += 1
                    return slot.id
        
        raise ValueError(f"No available slot for request {request.id}")
    
    def _find_best_slot_balanced(self, request: TruckRequest) -> int:
        """Find best slot considering capacity balance"""
        # Try preferred slot first
        preferred_slot = self.time_slots[request.preferred_time]
        if preferred_slot.current_load < preferred_slot.capacity:
            preferred_slot.current_load += 1
            return preferred_slot.id
        
        # Find slot with minimum utilization ratio
        best_slot = None
        best_ratio = float('inf')
        best_distance = float('inf')
        
        for slot in self.time_slots:
            if slot.current_load < slot.capacity:
                distance = abs(slot.id - request.preferred_time)
                utilization_ratio = slot.current_load / slot.capacity
                
                # Prioritize closer slots, break ties by utilization
                if (distance < best_distance or 
                    (distance == best_distance and utilization_ratio < best_ratio)):
                    best_slot = slot
                    best_distance = distance
                    best_ratio = utilization_ratio
        
        if best_slot:
            best_slot.current_load += 1
            return best_slot.id
        
        raise ValueError(f"No available slot for request {request.id}")

print("Enhanced EDF class defined with multiple strategies")

Enhanced EDF class defined with multiple strategies


In [ ]:
try:
    # Compare different EDF strategies
    def compare_edf_strategies():
        """Compare different EDF strategies on all test instances"""
        
        print("\n" + "=" * 80)
        print("EDF STRATEGY COMPARISON")
        print("=" * 80)
        
        results = {}
        
        for instance_name, (requests, time_slots) in test_instances.items():
            print(f"\n--- Instance: {instance_name.upper()} ---")
            print(f"Requests: {len(requests)}, Time Slots: {len(time_slots)}")
            
            enhanced_edf = EnhancedEDF(requests, time_slots)
            instance_results = {}
            
            for strategy in ['basic', 'weighted', 'capacity_balanced']:
                solution = enhanced_edf.solve(strategy)
                instance_results[strategy] = solution
                
                print(f"\n{strategy.title()} Strategy:")
                print(f"  Total Deviation: {solution['total_deviation']}")
                print(f"  Weighted Deviation: {solution['weighted_deviation']:.2f}")
                print(f"  Execution Time: {solution['execution_time']:.6f} sec")
            
            results[instance_name] = instance_results
            
            # Find best strategy for this instance
            best_strategy = min(instance_results.keys(), 
                               key=lambda s: instance_results[s]['weighted_deviation'])
            print(f"\nBest strategy for {instance_name}: {best_strategy}")
            print(f"Best weighted deviation: {instance_results[best_strategy]['weighted_deviation']:.2f}")
        
        return results
    
    strategy_results = compare_edf_strategies()
except Exception as e:
    print(f'Error in cell: {e}')


EDF STRATEGY COMPARISON

--- Instance: CONCRETE ---
Requests: 6, Time Slots: 4

Basic Strategy:
  Total Deviation: 0
  Weighted Deviation: 0.00
  Execution Time: 0.000031 sec

Weighted Strategy:
  Total Deviation: 0
  Weighted Deviation: 0.00
  Execution Time: 0.000008 sec

Capacity_Balanced Strategy:
  Total Deviation: 0
  Weighted Deviation: 0.00
  Execution Time: 0.000006 sec

Best strategy for concrete: basic
Best weighted deviation: 0.00

--- Instance: LARGE ---
Requests: 12, Time Slots: 6

Basic Strategy:
  Total Deviation: 1
  Weighted Deviation: 1.80
  Execution Time: 0.000012 sec

Weighted Strategy:
  Total Deviation: 1
  Weighted Deviation: 0.90
  Execution Time: 0.000011 sec

Capacity_Balanced Strategy:
  Total Deviation: 1
  Weighted Deviation: 1.80
  Execution Time: 0.000012 sec

Best strategy for large: weighted
Best weighted deviation: 0.90

--- Instance: STRESS ---
Requests: 9, Time Slots: 3
Error in cell: No available slot for request 7


In [ ]:
try:
    # Comprehensive visualization and analysis
    def visualize_edf_performance():
        """Create comprehensive visualizations of EDF performance"""
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('EDF Heuristic Performance Analysis', fontsize=16, fontweight='bold')
        
        # 1. Assignment Matrix for Concrete Example
        ax1 = axes[0, 0]
        requests, time_slots = test_instances['concrete']
        solution = concrete_solution
        
        assignment_matrix = [[0 for _ in time_slots] for _ in requests]
        for req in requests:
            assigned_slot = solution['assignments'][req.id]
            assignment_matrix[req.id-1][assigned_slot] = 1
        
        sns.heatmap(assignment_matrix, annot=True, cmap='RdYlGn_r', cbar=False,
                   xticklabels=[f'Slot {t.id}' for t in time_slots],
                   yticklabels=[f'Req {r.id}' for r in requests],
                   ax=ax1)
        ax1.set_title('Assignment Matrix\n(Concrete Example)', fontweight='bold')
        ax1.set_xlabel('Time Slots')
        ax1.set_ylabel('Truck Requests')
        
        # 2. Strategy Comparison - Weighted Deviation
        ax2 = axes[0, 1]
        instances = list(strategy_results.keys())
        strategies = ['basic', 'weighted', 'capacity_balanced']
        
        x = np.arange(len(instances))
        width = 0.25
        
        for i, strategy in enumerate(strategies):
            deviations = [strategy_results[inst][strategy]['weighted_deviation'] for inst in instances]
            ax2.bar(x + i*width, deviations, width, label=strategy.title(), alpha=0.8)
        
        ax2.set_xlabel('Test Instances')
        ax2.set_ylabel('Weighted Deviation')
        ax2.set_title('Strategy Comparison\n(Weighted Deviation)', fontweight='bold')
        ax2.set_xticks(x + width)
        ax2.set_xticklabels(instances, rotation=45)
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Execution Time Comparison
        ax3 = axes[0, 2]
        for i, strategy in enumerate(strategies):
            times = [strategy_results[inst][strategy]['execution_time'] * 1000 for inst in instances]
            ax3.bar(x + i*width, times, width, label=strategy.title(), alpha=0.8)
        
        ax3.set_xlabel('Test Instances')
        ax3.set_ylabel('Execution Time (ms)')
        ax3.set_title('Strategy Comparison\n(Execution Time)', fontweight='bold')
        ax3.set_xticks(x + width)
        ax3.set_xticklabels(instances, rotation=45)
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Deviation Distribution
        ax4 = axes[1, 0]
        deviations = []
        for req in concrete_requests:
            assigned_slot = concrete_solution['assignments'][req.id]
            deviation = abs(assigned_slot - req.preferred_time)
            deviations.append(deviation)
        
        ax4.hist(deviations, bins=range(max(deviations)+2), alpha=0.7, color='skyblue', edgecolor='black')
        ax4.set_xlabel('Deviation from Preferred Time')
        ax4.set_ylabel('Number of Requests')
        ax4.set_title('Deviation Distribution\n(Concrete Example)', fontweight='bold')
        ax4.grid(True, alpha=0.3)
        
        # Add statistics
        mean_dev = np.mean(deviations)
        ax4.axvline(mean_dev, color='red', linestyle='--', label=f'Mean: {mean_dev:.2f}')
        ax4.legend()
        
        # 5. Time Slot Utilization
        ax5 = axes[1, 1]
        utilization_data = []
        for instance_name in instances:
            requests, time_slots = test_instances[instance_name]
            solution = strategy_results[instance_name]['basic']
            
            for slot in time_slots:
                assigned = sum(1 for req in requests if solution['assignments'][req.id] == slot.id)
                utilization = assigned / slot.capacity * 100
                utilization_data.append({'instance': instance_name, 'slot': slot.id, 'utilization': utilization})
        
        util_df = pd.DataFrame(utilization_data)
        sns.boxplot(data=util_df, x='instance', y='utilization', ax=ax5)
        ax5.set_xlabel('Test Instances')
        ax5.set_ylabel('Utilization (%)')
        ax5.set_title('Time Slot Utilization\nAcross Instances', fontweight='bold')
        ax5.grid(True, alpha=0.3)
        
        # 6. Scalability Analysis
        ax6 = axes[1, 2]
        problem_sizes = [6, 12, 20, 30, 50, 100]
        execution_times = []
        
        for size in problem_sizes:
            # Generate random problem
            test_requests = [TruckRequest(i, np.random.randint(0, min(10, size//2)), 
                                          np.random.uniform(0.5, 2.0)) for i in range(size)]
            test_slots = [TimeSlot(j, max(2, size//10)) for j in range(min(10, size//2))]
            
            # Time the execution
            start = time.time()
            edf = EarliestDeadlineFirst(test_requests, test_slots)
            edf.solve()
            exec_time = time.time() - start
            execution_times.append(exec_time * 1000)
        
        ax6.loglog(problem_sizes, execution_times, 'o-', color='darkblue', linewidth=2, markersize=8)
        ax6.set_xlabel('Problem Size (Number of Requests)')
        ax6.set_ylabel('Execution Time (ms)')
        ax6.set_title('Scalability Analysis\n(EDF Performance)', fontweight='bold')
        ax6.grid(True, alpha=0.3)
        
        # Add complexity reference line
        x_ref = np.array(problem_sizes)
        y_ref = x_ref * np.log(x_ref) * 0.001  # O(n log n) reference
        ax6.plot(x_ref, y_ref, '--', color='red', label='O(n log n) reference')
        ax6.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print detailed analysis
        print("\n" + "=" * 80)
        print("DETAILED PERFORMANCE ANALYSIS")
        print("=" * 80)
        
        print("\n1. Algorithm Complexity:")
        print("   - Sorting requests: O(n log n)")
        print("   - Assignment process: O(n × m) where m = number of time slots")
        print("   - Overall complexity: O(n log n + n × m)")
        print("   - Space complexity: O(n + m)")
        
        print("\n2. Solution Quality Analysis:")
        for instance_name in instances:
            basic_dev = strategy_results[instance_name]['basic']['weighted_deviation']
            weighted_dev = strategy_results[instance_name]['weighted']['weighted_deviation']
            balanced_dev = strategy_results[instance_name]['capacity_balanced']['weighted_deviation']
            
            improvement_basic = 0
            improvement_weighted = ((basic_dev - weighted_dev) / basic_dev * 100) if basic_dev > 0 else 0
            improvement_balanced = ((basic_dev - balanced_dev) / basic_dev * 100) if basic_dev > 0 else 0
            
            print(f"   {instance_name.title()}:")
            print(f"     Basic: {basic_dev:.2f}")
            print(f"     Weighted improvement: {improvement_weighted:+.1f}%")
            print(f"     Balanced improvement: {improvement_balanced:+.1f}%")
    
    visualize_edf_performance()
except Exception as e:
    print(f'Error in cell: {e}')

In [ ]:
try:
    # Sensitivity analysis and what-if scenarios
    def sensitivity_analysis():
        """Perform sensitivity analysis on key parameters"""
        
        print("\n" + "=" * 80)
        print("SENSITIVITY ANALYSIS & WHAT-IF SCENARIOS")
        print("=" * 80)
        
        # Base instance
        base_requests, base_time_slots = test_instances['concrete']
        
        # Scenario 1: Vary capacity constraints
        print("\n1. Capacity Constraint Sensitivity:")
        print("   Capacity Multiplier → Weighted Deviation")
        
        capacity_multipliers = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
        for mult in capacity_multipliers:
            modified_slots = [TimeSlot(s.id, int(s.capacity * mult)) for s in base_time_slots]
            edf = EarliestDeadlineFirst(base_requests, modified_slots)
            solution = edf.solve()
            print(f"   {mult:>4.2f} → {solution['weighted_deviation']:>8.2f}")
        
        # Scenario 2: Vary request distribution
        print("\n2. Request Distribution Sensitivity:")
        print("   Distribution Type → Weighted Deviation")
        
        distributions = {
            'uniform': lambda n: [TruckRequest(i, np.random.randint(0, 4), 1.0) for i in range(n)],
            'concentrated': lambda n: [TruckRequest(i, np.random.choice([0, 1]), 1.0) for i in range(n)],
            'spread': lambda n: [TruckRequest(i, np.random.randint(0, 8), 1.0) for i in range(n)]
        }
        
        for dist_name, dist_func in distributions.items():
            test_requests = dist_func(8)
            test_slots = [TimeSlot(i, 2) for i in range(4)]
            edf = EarliestDeadlineFirst(test_requests, test_slots)
            solution = edf.solve()
            print(f"   {dist_name:>11} → {solution['weighted_deviation']:>8.2f}")
        
        # Scenario 3: Real-time performance test
        print("\n3. Real-time Performance Test:")
        print("   Problem Size → Avg Execution Time (ms)")
        
        sizes = [10, 25, 50, 100, 250, 500, 1000]
        trials = 10
        
        for size in sizes:
            times = []
            for _ in range(trials):
                test_requests = [TruckRequest(i, np.random.randint(0, min(20, size//5)), 
                                              np.random.uniform(0.5, 2.0)) for i in range(size)]
                test_slots = [TimeSlot(j, max(3, size//20)) for j in range(min(20, size//5))]
                
                start = time.time()
                edf = EarliestDeadlineFirst(test_requests, test_slots)
                edf.solve()
                times.append((time.time() - start) * 1000)
            
            avg_time = np.mean(times)
            print(f"   {size:>4} → {avg_time:>8.2f}")
            
            if avg_time > 1000:  # If超过1 second
                print(f"      ⚠ Performance degrades significantly at size {size}")
                break
        
        print("\n4. Comparison with Optimal (Small Instance):")
        # Create a small instance where we can compute optimal solution
        small_requests = [
            TruckRequest(1, 0, 1.0), TruckRequest(2, 0, 1.0), TruckRequest(3, 1, 1.0)
        ]
        small_slots = [TimeSlot(0, 2), TimeSlot(1, 1)]
        
        # EDF solution
        edf = EarliestDeadlineFirst(small_requests, small_slots)
        edf_solution = edf.solve()
        
        # Brute force optimal
        best_deviation = float('inf')
        for r1_slot in [0, 1]:
            for r2_slot in [0, 1]:
                for r3_slot in [0, 1]:
                    # Check capacity
                    slot0_count = sum(1 for s in [r1_slot, r2_slot, r3_slot] if s == 0)
                    slot1_count = sum(1 for s in [r1_slot, r2_slot, r3_slot] if s == 1)
                    
                    if slot0_count <= 2 and slot1_count <= 1:
                        deviation = (abs(r1_slot - 0) + abs(r2_slot - 0) + abs(r3_slot - 1))
                        best_deviation = min(best_deviation, deviation)
        
        gap = (edf_solution['total_deviation'] - best_deviation) / best_deviation * 100 if best_deviation > 0 else 0
        print(f"   EDF Deviation: {edf_solution['total_deviation']}")
        print(f"   Optimal Deviation: {best_deviation}")
        print(f"   Optimality Gap: {gap:.1f}%")
    
    sensitivity_analysis()
except Exception as e:
    print(f'Error in cell: {e}')


SENSITIVITY ANALYSIS & WHAT-IF SCENARIOS

1. Capacity Constraint Sensitivity:
   Capacity Multiplier → Weighted Deviation
Error in cell: No available slot for request 3


### Why this Tier exists vs earlier Tiers
This Tier 2 provides a practical heuristic approach that addresses the computational limitations of the mathematical formulation:

- **Computational efficiency**: O(n log n) complexity vs exponential for MIP
- **Real-time capability**: Suitable for dynamic gate operations
- **Implementation simplicity**: Easy to understand and deploy
- **Scalability**: Handles large instances that MIP cannot solve
- **Predictable performance**: Consistent execution times

### Pros / Cons vs Tier 1
**Pros:**
- Fast execution (milliseconds vs minutes/hours)
- Handles large problem instances
- Simple implementation and maintenance
- Suitable for real-time decision making
- No specialized software required

**Cons:**
- No optimality guarantees
- Solution quality depends on problem structure
- Limited optimization capability
- May get stuck in local optima
- No sensitivity analysis built-in

### When to use this Tier
- **Real-time operations** requiring immediate decisions
- **Large-scale problems** where MIP is infeasible
- **Resource-constrained environments** with limited computing power
- **Initial solution generation** for advanced methods
- **Dynamic environments** with frequent re-optimization needs
- **Production systems** requiring reliable, fast performance